# ExpresSo Deep EDA Add-on

Add-on cells for the hackathon notebook. These cells focus on:

- baseline-normalized lift (`store_id x category x day_of_week`)
- promo/event/holiday/weather effects without store/category bias
- forecast-window known signals for Nov-Dec 2024
- store service/capacity, customer behavior, SKU mix, and stockout diagnostics

Keep the leakage rule in mind: this notebook is for EDA. When turning ideas into model
features, lag/rolling features from sales/order/inventory must be shifted by horizon.



In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 12, "figure.figsize": (12, 5)})
plt.rcParams["font.family"] = ["DejaVu Sans", "Noto Sans Thai", "Tahoma", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")

BASE_CANDIDATES = [
    Path("/kaggle/input/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path("/kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path("./super-ai-engineer-season-6-coffee-chain-hackathon"),
]
BASE = next((p for p in BASE_CANDIDATES if p.exists()), None)
if BASE is None:
    raise FileNotFoundError("Could not find competition dataset. Update BASE_CANDIDATES.")

TRAIN = BASE / "train"
TEST = BASE / "test"
print("Using dataset:", BASE)




In [ ]:
txn = pd.read_csv(TRAIN / "TRANSACTION.csv")
order = pd.read_csv(TRAIN / "ORDER.csv", parse_dates=["date"])
prod = pd.read_csv(TRAIN / "PRODUCT.csv")
store = pd.read_csv(TRAIN / "STORE.csv", parse_dates=["opened_date"])
promo = pd.read_csv(TRAIN / "PROMOTION.csv", parse_dates=["start_date", "end_date"])
event = pd.read_csv(TRAIN / "LOCAL_EVENT.csv", parse_dates=["date"])
date_dim = pd.read_csv(TRAIN / "DATE_DIM.csv", parse_dates=["date"])
cust = pd.read_csv(TRAIN / "CUSTOMER.csv", parse_dates=["registration_date"])

test_date = pd.read_csv(TEST / "DATE_DIM.csv", parse_dates=["date"])
test_promo = pd.read_csv(TEST / "PROMOTION.csv", parse_dates=["start_date", "end_date"])
test_event = pd.read_csv(TEST / "LOCAL_EVENT.csv", parse_dates=["date"])

cat_order = [
    "Coffee",
    "Tea",
    "Chocolate & Milk",
    "Juice & Smoothie",
    "Bakery",
    "Savory Bakery",
    "Merchandise",
]

line = (
    txn.merge(order[["order_id", "store_id", "date", "hour", "customer_id", "is_member", "payment_method"]], on="order_id", how="left")
       .merge(prod[["product_id", "product_name", "category", "serve_type", "base_price", "is_seasonal", "is_limited_edition"]], on="product_id", how="left")
)

daily = (
    line.groupby(["store_id", "category", "date"], observed=True)
        .agg(
            units_sold=("units_sold", "sum"),
            revenue=("revenue", "sum"),
            n_orders=("order_id", "nunique"),
            n_customers=("customer_id", "nunique"),
        )
        .reset_index()
)

all_dates = pd.date_range(order["date"].min(), order["date"].max(), freq="D")
idx = pd.MultiIndex.from_product(
    [sorted(store["store_id"].unique()), cat_order, all_dates],
    names=["store_id", "category", "date"],
)
daily = (
    daily.set_index(["store_id", "category", "date"])
         .reindex(idx, fill_value=0)
         .reset_index()
         .merge(store, on="store_id", how="left")
         .merge(date_dim, on="date", how="left")
)

daily["dow_baseline"] = daily.groupby(["store_id", "category", "day_of_week"])["units_sold"].transform("mean")
daily["units_rel"] = np.where(daily["dow_baseline"] > 0, daily["units_sold"] / daily["dow_baseline"], 0)

print("line:", line.shape, "| daily:", daily.shape)
print(daily[["units_sold", "units_rel"]].describe(percentiles=[.5, .75, .9, .95, .99]))




In [ ]:
def lift_table(df, flag, by="category", value="units_rel"):
    """Return off/on means and lift percent for a boolean flag."""
    out = (
        df.groupby([by, flag], observed=True)[value]
          .mean()
          .unstack(flag)
          .rename(columns={False: "off", True: "on", 0: "off", 1: "on"})
    )
    for col in ["off", "on"]:
        if col not in out:
            out[col] = np.nan
    out["lift_pct"] = (out["on"] - out["off"]) / out["off"] * 100
    return out.sort_values("lift_pct", ascending=False)


def plot_lift(df, title, ax=None):
    ax = ax or plt.gca()
    df["lift_pct"].plot(kind="barh", ax=ax, color=np.where(df["lift_pct"] >= 0, "#2a9d8f", "#e76f51"))
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("normalized lift %")
    ax.invert_yaxis()


def expand_promo(promo_df, product_df):
    p = promo_df.merge(product_df[["product_id", "category"]], on="product_id", how="left").copy()
    p["date"] = p.apply(lambda r: pd.date_range(r["start_date"], r["end_date"], freq="D"), axis=1)
    p = p.explode("date")
    return (
        p.groupby(["store_id", "category", "date"], observed=True)
         .agg(
             has_promo=("promo_id", "count"),
             max_discount=("discount_pct", "max"),
             promo_type=("promo_type", lambda s: "|".join(sorted(set(s)))),
             email_sent=("email_sent", "max"),
             social_campaign=("social_campaign", "max"),
             promo_product_count=("product_id", "nunique"),
         )
         .reset_index()
    )


def heatmap_table(df, row, col, value, min_count=10):
    count = df.groupby([row, col], observed=True)[value].size().unstack(fill_value=0)
    mean = df.groupby([row, col], observed=True)[value].mean().unstack()
    return mean.where(count >= min_count)




In [ ]:
flags = ["is_weekend", "is_holiday", "is_payday", "is_school_break", "is_rainy_season"]
fig, axes = plt.subplots(1, len(flags), figsize=(22, 5), sharex=False)
for flag, ax in zip(flags, axes):
    tab = lift_table(daily, flag)
    plot_lift(tab, flag, ax=ax)
plt.tight_layout()
plt.show()

for flag in flags:
    display(lift_table(daily, flag).round(3))




In [ ]:
holiday_lift = (
    daily[daily["is_holiday"]]
    .groupby(["holiday_name", "category"], observed=True)
    .agg(rel_mean=("units_rel", "mean"), raw_mean=("units_sold", "mean"), dates=("date", "nunique"))
    .reset_index()
    .sort_values("rel_mean", ascending=False)
)
display(holiday_lift.head(30))

forecast_window = test_date[(test_date["date"] >= "2024-11-01") & (test_date["date"] <= "2024-12-31")].copy()
known_signal_arr = (
    np.where(forecast_window["is_holiday"], "holiday", "")
    + np.where(forecast_window["is_payday"], "|payday", "")
    + np.where(forecast_window["is_weekend"], "|weekend", "")
)
forecast_window["known_signal"] = pd.Series(known_signal_arr, index=forecast_window.index).str.strip("|")

plt.figure(figsize=(16, 3))
signal_rank = {"": 0, "weekend": 1, "payday": 2, "payday|weekend": 3, "holiday": 4, "holiday|weekend": 5}
tmp = forecast_window.assign(signal_score=forecast_window["known_signal"].map(signal_rank).fillna(1))
plt.scatter(tmp["date"], np.ones(len(tmp)), c=tmp["signal_score"], cmap="viridis", s=80)
for _, r in tmp[tmp["is_holiday"] | tmp["is_payday"]].iterrows():
    plt.text(r["date"], 1.03, r["date"].strftime("%m-%d"), rotation=90, ha="center", va="bottom", fontsize=8)
plt.yticks([])
plt.title("Known calendar signals in forecast window: Nov-Dec 2024")
plt.tight_layout()
plt.show()

display(forecast_window.loc[forecast_window["is_holiday"] | forecast_window["is_payday"], ["date", "day_of_week", "is_holiday", "holiday_name", "is_payday"]])




In [ ]:
store_profile = (
    daily.groupby(["store_id", "neighborhood_type"], observed=True)
    .agg(units_per_day=("units_sold", "sum"), rel_mean=("units_rel", "mean"))
    .reset_index()
)
store_profile["units_per_day"] /= daily["date"].nunique()

store_feat = store.copy()
store_feat["open_hour"] = store_feat["open_time"].str.slice(0, 2).astype(int) + store_feat["open_time"].str.slice(3, 5).astype(int) / 60
store_feat["close_hour"] = store_feat["close_time"].str.slice(0, 2).astype(int) + store_feat["close_time"].str.slice(3, 5).astype(int) / 60
store_feat["operating_hours"] = store_feat["close_hour"] - store_feat["open_hour"]
store_feat["store_age_days"] = (daily["date"].max() - store_feat["opened_date"]).dt.days

store_profile = store_profile.merge(store_feat, on=["store_id", "neighborhood_type"], how="left")
store_profile["units_per_staff_day"] = store_profile["units_per_day"] / store_profile["staff_count"]
store_profile["units_per_seat_day"] = store_profile["units_per_day"] / store_profile["seating_capacity"]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for x, ax in zip(["seating_capacity", "staff_count", "operating_hours", "store_age_days"], axes):
    sns.scatterplot(data=store_profile, x=x, y="units_per_day", hue="neighborhood_type", s=90, ax=ax)
    for _, r in store_profile.iterrows():
        ax.text(r[x], r["units_per_day"], str(r["store_id"]), fontsize=8)
    ax.legend([], [], frameon=False)
plt.tight_layout()
plt.show()

display(store_profile.sort_values("units_per_day", ascending=False))




In [ ]:
promo_daily = expand_promo(promo, prod)
daily_promo = daily.merge(promo_daily, on=["store_id", "category", "date"], how="left")
daily_promo["has_promo"] = daily_promo["has_promo"].fillna(0).gt(0)
daily_promo["max_discount"] = daily_promo["max_discount"].fillna(0)

promo_lift = lift_table(daily_promo, "has_promo")
display(promo_lift.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_lift(promo_lift, "Promo normalized lift by category", ax=axes[0])
sns.boxplot(data=daily_promo, x="category", y="units_rel", hue="has_promo", ax=axes[1], showfliers=False)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_xticklabels(axes[1].get_xticklabels(), ha="right")
axes[1].set_title("Distribution of normalized demand: promo vs no promo")
plt.tight_layout()
plt.show()

promo_type_long = (
    daily_promo[daily_promo["has_promo"]]
    .assign(promo_type=daily_promo["promo_type"].fillna("").str.split("|"))
    .explode("promo_type")
)
promo_type_values = sorted(v for v in promo_type_long["promo_type"].dropna().unique() if v)
promo_type_code_map = {v: f"promo_type_{i+1}" for i, v in enumerate(promo_type_values)}
promo_type_long["promo_type_display"] = promo_type_long["promo_type"].map(promo_type_code_map).fillna("promo_type_other")
promo_type_heat = heatmap_table(promo_type_long, "promo_type_display", "category", "units_rel", min_count=20)
plt.figure(figsize=(13, 6))
sns.heatmap(promo_type_heat, annot=True, fmt=".2f", cmap="RdYlGn", center=1)
plt.title("Normalized demand by promo_type x category")
plt.xlabel("category")
plt.ylabel("promo_type_code")
plt.tight_layout()
plt.show()
display(
    pd.DataFrame(
        {
            "promo_type_code": list(promo_type_code_map.values()),
            "promo_type_raw": list(promo_type_code_map.keys()),
        }
    )
)

forecast_promo_daily = expand_promo(test_promo, prod)
train_rate = (
    promo_daily[promo_daily["date"].between(order["date"].min(), order["date"].max())]
    .groupby("category", observed=True)
    .size()
    / (store["store_id"].nunique() * daily["date"].nunique())
)
test_rate = (
    forecast_promo_daily[forecast_promo_daily["date"].between("2024-11-01", "2024-12-31")]
    .groupby("category", observed=True)
    .size()
    / (store["store_id"].nunique() * 61)
)
promo_drift = pd.concat([train_rate.rename("train_promo_rate"), test_rate.rename("forecast_promo_rate")], axis=1).fillna(0)
promo_drift["pp_change"] = (promo_drift["forecast_promo_rate"] - promo_drift["train_promo_rate"]) * 100
display((promo_drift * 100).round(2))

drink_promo_dates = daily_promo.loc[
    daily_promo["has_promo"] & daily_promo["category"].isin(["Coffee", "Tea"]),
    ["store_id", "date"],
].drop_duplicates()
drink_promo_dates["drink_promo_active"] = True
merch = daily_promo[daily_promo["category"].eq("Merchandise")].merge(drink_promo_dates, on=["store_id", "date"], how="left")
merch["drink_promo_active"] = merch["drink_promo_active"].fillna(False)
display(lift_table(merch, "drink_promo_active", by="neighborhood_type").round(3))




In [ ]:
event_daily = (
    event.groupby(["store_id", "date"], observed=True)
    .agg(event_count=("event_id", "count"), event_type=("event_type", lambda s: "|".join(sorted(set(s)))))
    .reset_index()
)
daily_event = daily.merge(event_daily, on=["store_id", "date"], how="left")
daily_event["has_event"] = daily_event["event_count"].fillna(0).gt(0)

event_lift = lift_table(daily_event, "has_event")
display(event_lift.round(3))

event_long = (
    daily_event[daily_event["has_event"]]
    .assign(event_type=daily_event["event_type"].fillna("").str.split("|"))
    .explode("event_type")
)
event_type_heat = heatmap_table(event_long, "event_type", "category", "units_rel", min_count=10)
plt.figure(figsize=(13, 6))
sns.heatmap(event_type_heat, annot=True, fmt=".2f", cmap="RdYlGn", center=1)
plt.title("Normalized demand by event_type x category")
plt.tight_layout()
plt.show()

forecast_events = test_event[test_event["date"].between("2024-11-01", "2024-12-31")]
display(forecast_events.groupby("event_type").size().rename("forecast_event_count").sort_values(ascending=False))
display(forecast_events.sort_values(["date", "store_id"]).head(40))




In [ ]:
hour_profile = (
    line.groupby(["hour"], observed=True)
    .agg(orders=("order_id", "nunique"), units=("units_sold", "sum"), revenue=("revenue", "sum"))
    .reset_index()
)
hour_profile["units_per_order"] = hour_profile["units"] / hour_profile["orders"]
hour_profile["order_share"] = hour_profile["orders"] / hour_profile["orders"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=hour_profile, x="hour", y="order_share", ax=axes[0], color="#457b9d")
axes[0].set_title("Order share by hour")
sns.lineplot(data=hour_profile, x="hour", y="units_per_order", marker="o", ax=axes[1], color="#e76f51")
axes[1].set_title("Units/order by hour")
plt.tight_layout()
plt.show()

nb_hour = (
    line.merge(store[["store_id", "neighborhood_type"]], on="store_id", how="left")
    .groupby(["neighborhood_type", "hour"], observed=True)["order_id"]
    .nunique()
    .reset_index(name="orders")
)
top_hours = nb_hour.sort_values(["neighborhood_type", "orders"], ascending=[True, False]).groupby("neighborhood_type").head(3)
display(top_hours)

basket = (
    line.groupby("order_id", observed=True)
    .agg(
        basket=("category", lambda s: " + ".join(sorted(set(s)))),
        n_categories=("category", "nunique"),
        units=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        customer_id=("customer_id", "first"),
        payment_method=("payment_method", "first"),
    )
    .reset_index()
)
basket["segment"] = np.where(basket["customer_id"].isna(), "walk_in", "member")
display(basket["basket"].value_counts().head(15).rename_axis("basket").reset_index(name="orders"))
display(basket.groupby("segment").agg(orders=("order_id", "count"), units_per_order=("units", "mean"), revenue_per_order=("revenue", "mean")))
display(basket.groupby("payment_method").agg(orders=("order_id", "count"), units_per_order=("units", "mean"), revenue_per_order=("revenue", "mean")).sort_values("orders", ascending=False))




In [ ]:
sku_summary = (
    line.groupby(["product_id", "product_name", "category", "serve_type", "base_price", "is_seasonal", "is_limited_edition"], observed=True)
    .agg(units=("units_sold", "sum"), revenue=("revenue", "sum"), avg_discount=("discount_applied", "mean"))
    .reset_index()
    .sort_values("units", ascending=False)
)
display(sku_summary.head(20))

seasonal_share = (
    line.groupby(["category", "is_seasonal"], observed=True, as_index=False)["units_sold"]
    .sum()
)
seasonal_share["unit_share"] = seasonal_share["units_sold"] / seasonal_share.groupby("category", observed=True)["units_sold"].transform("sum")
display(
    seasonal_share[seasonal_share["is_seasonal"]]
    .sort_values("unit_share", ascending=False)
)

monthly = daily.assign(year=daily["date"].dt.year, month=daily["date"].dt.month)
yoy = (
    monthly[monthly["month"].le(10)]
    .groupby(["year", "category"], observed=True)["units_sold"]
    .mean()
    .unstack("year")
)
yoy["yoy_pct"] = (yoy[2024] - yoy[2023]) / yoy[2023] * 100
display(yoy.sort_values("yoy_pct", ascending=False).round(2))

recent = monthly[monthly["date"].between("2024-05-01", "2024-10-31")].copy()
recent["period"] = np.where(recent["date"].dt.month <= 7, "May-Jul", "Aug-Oct")
recent_tab = recent.groupby(["category", "period"], observed=True)["units_sold"].mean().unstack()
recent_tab["change_pct"] = (recent_tab["Aug-Oct"] - recent_tab["May-Jul"]) / recent_tab["May-Jul"] * 100
display(recent_tab.sort_values("change_pct", ascending=False).round(2))




In [ ]:
inventory = pd.read_csv(TRAIN / "INVENTORY.csv", parse_dates=["date"])
txn_sku_day = (
    line.groupby(["store_id", "product_id", "date"], observed=True)["units_sold"]
    .sum()
    .reset_index(name="txn_units_sold")
)
inv_cmp = inventory.merge(txn_sku_day, on=["store_id", "product_id", "date"], how="left")
inv_cmp["txn_units_sold"] = inv_cmp["txn_units_sold"].fillna(0)
inv_cmp = inv_cmp.merge(prod[["product_id", "category"]], on="product_id", how="left")
inv_cmp["abs_diff"] = (inv_cmp["units_sold"] - inv_cmp["txn_units_sold"]).abs()

print("Inventory exact match rate:", (inv_cmp["units_sold"].eq(inv_cmp["txn_units_sold"]).mean()).round(4))
print("Inventory MAE vs transaction SKU-store-day:", inv_cmp["abs_diff"].mean().round(3))
display(inv_cmp.groupby("category").agg(stockout_rate=("is_stockout", "mean"), inv_mae=("abs_diff", "mean")).sort_values("stockout_rate", ascending=False))
display(inv_cmp.groupby("store_id").agg(stockout_rate=("is_stockout", "mean"), inv_mae=("abs_diff", "mean")).sort_values("stockout_rate", ascending=False).head(20))

stockout_cell = (
    inv_cmp[inv_cmp["is_stockout"]]
    .groupby(["store_id", "category", "date"], observed=True)
    .size()
    .reset_index(name="stockout_sku_count")
)
daily_stock = daily.merge(stockout_cell, on=["store_id", "category", "date"], how="left")
daily_stock["any_stockout"] = daily_stock["stockout_sku_count"].fillna(0).gt(0)
display(lift_table(daily_stock, "any_stockout").round(3))




In [ ]:
feature_map = pd.DataFrame(
    [
        ["lag/rolling", "lag_7/14/28/35/60/90, rolling shifted by horizon", "must be horizon-safe"],
        ["baseline", "store_category_dow_mean, y_rel diagnostics", "separate structural volume from shocks"],
        ["store", "neighborhood_type, capacity, staff, hours, drive_through, store_age", "location/service/capacity"],
        ["calendar", "holiday_name, payday, school_break, rainy_season, month, DOW", "known future lookup"],
        ["promo", "has_promo, max_discount, promo_type, channel flags, promo_product_count", "known future lookup"],
        ["cross_promo", "drink_promo_active for Merchandise/Bakery targets", "cannibalization/cross-sell"],
        ["event", "has_event, event_type, event_count, store_type x event_type", "local demand shocks"],
        ["stockout", "lag_stockout_rate, stockout_sku_count, days_since_stockout", "recorded demand censoring"],
        ["customer", "rolling member ratio, basket attachment, payment mix", "behavior proxy, no review text"],
        ["traffic_proxy", "peak-hour share, morning/evening share by store type", "commute/location behavior"],
        ["sku_mix", "seasonal_unit_share, limited_sku_count, top_sku_promo/stockout", "hidden category composition"],
        ["drift", "recent 28/56/90 trend, YoY same month, forecast promo/event density", "public/private robustness"],
    ],
    columns=["block", "feature ideas", "purpose"],
)
display(feature_map)
